# 02 — Q3 Fact Overlay, Q4 Why-not Universe, Q5 Frontier Trace

Three diagnostic capabilities that answer increasingly substrate-level questions:

1. **Q3 Fact Overlay** (`check_fact_overlay_binding`) — *"if this fact were different, would the binding hold?"* Returns a `before / after / diff` triple. The ledger is **not** written; the overlay is a pure projection-merge.
2. **Q4 Why-not Universe** (`check_why_not_universe`) — given an explicit finite candidate universe, partition it into `green` (binding holds) and `red` (binding fails) rows, with a per-row diagnostic locator.
3. **Q5 Evaluator Frontier Trace** (`evaluate_native_where_frontier`) — drop into the evaluator layer with a where-body and the projected fact view, and read the per-branch frontier where evaluation collapsed.

**Prerequisites:** [01_sdk_check_diagnose.ipynb](01_sdk_check_diagnose.ipynb) for the SDK / Check / Diagnose foundation. **Next:** [03_proofframe_rule_overlays.ipynb](03_proofframe_rule_overlays.ipynb).

## Setup

Same fixture shape as chapter 1. The fixture is rebuilt inline so this notebook stands on its own.

In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from kernel.application import (
    build_fact_value_override,
    build_frontier_view_facts,
    build_schema_index,
    build_why_not_candidate_universe,
    check_fact_overlay_binding,
    check_why_not_universe,
    entity_info,
    field_predicate,
    resolve_selector,
)
from kernel.application.protocol import (
    CompiledDerivationPlan,
    CompiledHeadCall,
    EntitySelector,
    EvaluationOverlay,
    FactOverlayCheckRequest,
    FieldPath,
    WhyNotUniverseRequest,
)
from kernel.core.evidence.write_protocol import set_field
from kernel.core.rules.frontier import evaluate_native_where_frontier
from kernel.core.store import Store
from kernel.sdk import Entity, Field, Identity, compile_schema_from_classes


class Person(Entity):
    name: str = Identity(primary_key=True)
    age: int = Field(cardinality='single')
    region: str = Field(cardinality='single')


@dataclass(frozen=True)
class SeededPerson:
    e_ref: str
    age: int
    region: str
    age_asrt_id: str
    age_pred_id: str
    region_pred_id: str
    exists_pred_id: str


def seed_person(store, index, *, name, age, region):
    info = entity_info(index, 'Person')
    ref = resolve_selector(
        EntitySelector(entity_type='Person', identity={'name': name}),
        index=index,
    )
    encoded = ref.encoded_ref or ''
    set_field(store.ledger, info.exists_predicate_id, encoded, [])
    set_field(store.ledger, info.identity_predicates['name'].pred_id, encoded, [('string', name)])
    age_pred = field_predicate(index, 'Person', 'age').pred_id
    region_pred = field_predicate(index, 'Person', 'region').pred_id
    age_asrt_id = set_field(store.ledger, age_pred, encoded, [('int', age)])
    set_field(store.ledger, region_pred, encoded, [('string', region)])
    return SeededPerson(
        e_ref=encoded,
        age=age,
        region=region,
        age_asrt_id=age_asrt_id,
        age_pred_id=age_pred,
        region_pred_id=region_pred,
        exists_pred_id=info.exists_predicate_id,
    )


def _binding(*items):
    return tuple(sorted(items, key=lambda item: item[0]))


schema_ir = compile_schema_from_classes([Person])
index = schema_ir and build_schema_index(schema_ir)
store = Store(schema_ir)
people = {
    'alice': seed_person(store, index, name='alice', age=25, region='us'),
    'bob':   seed_person(store, index, name='bob',   age=30, region='eu'),
    'carol': seed_person(store, index, name='carol', age=28, region='us'),
}
person_info = entity_info(index, 'Person')
plan = CompiledDerivationPlan(
    derivation_id='round-story-person-snapshot',
    version='1.0',
    body_ir=[
        ('pred', person_info.exists_predicate_id, ['$p']),
        ('pred', field_predicate(index, 'Person', 'age').pred_id,    ['$p', '$age']),
        ('pred', field_predicate(index, 'Person', 'region').pred_id, ['$p', '$region']),
    ],
    heads=(CompiledHeadCall(
        target_pred_id=person_info.exists_predicate_id,
        head_var_names=('$p', '$age', '$region'),
    ),),
)
print(f'Seeded {len(people)} people; plan body atoms = {len(plan.body_ir)}.')

## 1. Q3 Fact Overlay — what if Alice were 30?

`check_fact_overlay_binding(...)` evaluates a binding twice — `before` (raw ledger view) and `after` (ledger view with the overlay's `fact_actions` applied) — and returns a structured diff. The ledger is never written.

The overlay below uses `build_fact_value_override(...)` to construct a `FactValueOverride` that flips Alice's age from 25 to 30. We assert:

- `before.status == 'failed'` (Alice's actual age is 25)
- `after.status == 'passed'` (under the overlay she is 30)
- `diff.status_changed == True`
- The ledger byte-dump is identical before and after the call (immutability invariant).

In [ ]:
def _ledger_dump(store):
    return '\n'.join(store.ledger._get_connection().iterdump()).encode('utf-8')


alice = people['alice']
overlay = EvaluationOverlay(
    fact_actions=(
        build_fact_value_override(
            store,
            index,
            e_ref=alice.e_ref,
            field=FieldPath(entity_type='Person', field_name='age'),
            new_value=30,
            note='demo overlay: Alice turns 30',
        ),
    ),
)
binding_alice_30 = _binding(('$p', alice.e_ref), ('$age', 30), ('$region', 'us'))
ledger_before = _ledger_dump(store)

result = check_fact_overlay_binding(
    FactOverlayCheckRequest(
        plan=plan,
        binding=binding_alice_30,
        overlay=overlay,
        engine='native',
    ),
    store=store,
)

assert result.status == 'passed', result
assert result.before.status == 'failed'
assert result.after.status == 'passed'
assert result.diff.status_changed
assert _ledger_dump(store) == ledger_before, 'Fact Overlay must not write ledger'

print(f'overall.status      : {result.status}')
print(f'before.status       : {result.before.status}')
print(f'after.status        : {result.after.status}')
print(f'diff.status_changed : {result.diff.status_changed}')
print(f'diff.matched_count_delta: {result.diff.matched_count_delta}')
print('ledger byte-identical : True (overlay does not write the ledger)')

## 2. Q4 Why-not Universe — green / red board over an explicit candidate set

`check_why_not_universe(...)` takes a finite candidate universe — built with `build_why_not_candidate_universe(...)` from a list of var-binding dicts — and partitions it.

Below we ask three candidates, all targeting `(age=30, region=us)`:

- `(alice, 30, us)` — Alice's actual age is 25, so this fails at the age atom
- `(bob, 30, eu)`   — Bob actually is 30/eu, so this passes
- `(carol, 30, us)` — Carol's actual age is 28, so this fails at the age atom

Expected: 1 row in `green` (Bob), 2 rows in `red` (Alice, Carol). Each red row carries a `WhyNotRowDiagnostic` whose `atom_locator.failed_atom_index == 1`.

In [ ]:
bob = people['bob']
carol = people['carol']
alice_candidate = _binding(('$p', alice.e_ref), ('$age', 30), ('$region', 'us'))
bob_candidate   = _binding(('$p', bob.e_ref),   ('$age', 30), ('$region', 'eu'))
carol_candidate = _binding(('$p', carol.e_ref), ('$age', 30), ('$region', 'us'))

result = check_why_not_universe(
    WhyNotUniverseRequest(
        plan=plan,
        candidate_universe=build_why_not_candidate_universe(
            plan,
            (
                {'$p': alice.e_ref, '$age': 30, '$region': 'us'},
                {'$p': bob.e_ref,   '$age': 30, '$region': 'eu'},
                {'$p': carol.e_ref, '$age': 30, '$region': 'us'},
            ),
        ),
        engine='native',
    ),
    store=store,
)

assert result.status == 'completed'
assert result.green == (bob_candidate,)
assert tuple(row.binding for row in result.red) == (alice_candidate, carol_candidate)

print(f'status      : {result.status}')
print(f'green count : {len(result.green)} (expected 1: bob)')
print(f'red count   : {len(result.red)}  (expected 2: alice + carol)')
for row in result.red:
    diag = row.diagnostic
    locator = diag.atom_locator
    name = next(n for n, p in people.items() if p.e_ref == dict(row.binding)['$p'])
    print(
        f'  red {name:5s}: status={diag.status} '
        f'failure_kind={diag.failure_kind} '
        f'failed_atom_index={locator.failed_atom_index}'
    )

## 3. Q5 Evaluator Frontier Trace — substrate-layer view

`evaluate_native_where_frontier(view_facts, where)` is the substrate-layer probe. Given a where-body (the same `('pred', pid, args)` shape as the plan body) and a projected fact view (a dict `{pred_id: [tuple, ...]}`), it returns:

- `bindings` — the rows that survived the full where-body
- `frontier_rows` — the per-branch frontier where evaluation stopped

Below we ask the unsatisfiable where-body `exists($p) ∧ age($p, 99) ∧ region($p, 'us')` against the projected view of all three people. Expected: `bindings == []` and one frontier row at `branch_index=0, failed_atom_index=1, frontier_count=3, failure_kind='atom_filter_empty'` — three persons enter the age atom and all three are filtered out, collapsing the branch.

In [ ]:
any_person = next(iter(people.values()))
view_facts = build_frontier_view_facts(store)

where = [
    ('pred', any_person.exists_pred_id, ['$p']),
    ('pred', any_person.age_pred_id,    ['$p', 99]),
    ('pred', any_person.region_pred_id, ['$p', 'us']),
]

result = evaluate_native_where_frontier(view_facts, where)

assert result.bindings == []
assert len(result.frontier_rows) == 1
row = result.frontier_rows[0]
assert row.branch_index == 0
assert row.failed_atom_index == 1
assert row.frontier_count == 3
assert row.failure_kind == 'atom_filter_empty'

print(f'bindings              : {result.bindings}')
print(f'frontier rows count   : {len(result.frontier_rows)}')
print(f'  row.branch_index    = {row.branch_index}')
print(f'  row.failed_atom_index = {row.failed_atom_index}  (1 = age atom)')
print(f'  row.atoms_satisfied = {row.atoms_satisfied}')
print(f'  row.frontier_count  = {row.frontier_count}  (3 persons entered the age atom)')
print(f'  row.failure_kind    = {row.failure_kind!r}')

## Wrap-up

| Capability | API | Module |
|---|---|---|
| Q3 Fact Overlay   | `check_fact_overlay_binding(FactOverlayCheckRequest)` | `kernel.application.fact_overlay_runtime` |
| Q4 Why-not Universe | `check_why_not_universe(WhyNotUniverseRequest)`     | `kernel.application.why_not_runtime` |
| Q5 Frontier Trace | `evaluate_native_where_frontier(view_facts, where)`   | `kernel.core.rules.frontier` |

**Substrate vs application layer:** Q3 and Q4 are application-layer capabilities (they accept a `CompiledDerivationPlan` and run an engine). Q5 sits one level lower — it accepts a raw where-body and a projected fact view, and answers the structural question "where does the native where-body collapse?" without seeded bindings.

**Public-surface boundary (Batch 8):** Q3 / Q4 are advanced-importable from `kernel.application`. Q5 is advanced-importable from `kernel.core.rules.frontier`. v0.1 ships **no** SDK shells or service routes for any of the three.

**Next:** [03_proofframe_rule_overlays.ipynb](03_proofframe_rule_overlays.ipynb) — Batch 4 ProofFrame Rechecker + Batch 5a/b/c rule overlay actions (Disable / Literal Replace / Add Condition).